In [6]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Create map
Map = geemap.Map(center=[13.2, -59.5], zoom=10)

# Barbados boundary from GAUL
barbados = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Barbados'))
)

roads = ee.FeatureCollection("projects/deforestation-495419/assets/roads_lines")

# Rasterize roads
roads_raster = ee.Image().byte().paint(roads, 1)

# Distance to nearest road in meters
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_roads_m")
    .clip(barbados)
)

vis = {
    "min": 0,
    "max": 5000,
    "palette": ["white", "blue", "green", "yellow", "red"]
}

# Map.addLayer(barbados, {}, "Barbados boundary")

Map.addLayer(
    roads_raster,
    {"palette": ["black"]},
    "Roads (raster)"
)

Map.addLayer(
    distance_to_roads,
    vis,
    "Distance to Roads (m)"
)

Map 

Map(center=[13.2, -59.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…